# README

This .ipynb will use the new laboneq application library rather than creating each experiment anew using context managers.

The goal is to have this be more robust and easy to navigate, with fewer namespace collisions and fewer side-effect filled functions (dependent on external state)

# Preamble

In [1]:
%load_ext autoreload
%autoreload 2
    
from all_imports import *
from helper import *
from qelement_helper import *
from qops_helper import *
from qubit_experiments import *
from plot_helper import *
from analysis_helper import *

from laboneq.analysis.fitting import (
    lorentzian,
    oscillatory,
    oscillatory_decay,
    exponential_decay,
)

[DC1(YokogawaGS200)] Could not connect at TCPIP0:192.168.1.78::inst0::INSTR
Traceback (most recent call last):
  File "c:\Users\QNL\anaconda3\envs\gridium_ZI\Lib\site-packages\qcodes\instrument\visa.py", line 296, in _connect_and_handle_error
    visa_handle = self._open_resource(address, visalib)
  File "c:\Users\QNL\anaconda3\envs\gridium_ZI\Lib\site-packages\qcodes\instrument\visa.py", line 320, in _open_resource
    resource = resource_manager.open_resource(address)
  File "c:\Users\QNL\anaconda3\envs\gridium_ZI\Lib\site-packages\pyvisa\highlevel.py", line 3265, in open_resource
    info = self.resource_info(resource_name, extended=True)
  File "c:\Users\QNL\anaconda3\envs\gridium_ZI\Lib\site-packages\pyvisa\highlevel.py", line 3190, in resource_info
    ret, _err = self.visalib.parse_resource_extended(
                ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        self.session, resource_name
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\QNL\anaconda3\envs\gridium

VI_ERROR_RSRC_NFOUND (-1073807343): Insufficient location information or the requested device or resource is not present in the system.
Connected to: YOKOGAWA GS210 (serial:91P925819, firmware:2.02) in 0.11s


## ZI Connection

In [2]:
# Load in text file defining logical signal layout
with open('device_description/GKPv4_1_X2Y7.txt') as f:
    descriptor_shfqc = f.read()

# Define and Load our Device Setup
device_setup = DeviceSetup.from_descriptor(
    yaml_text=descriptor_shfqc, # yaml fully describes logical signal layout
    server_host="localhost",    # LabOne dataserver host name
    server_port="8004",         # port number of the dataserver - default is 8004
    setup_name="ersevim_host",  # setup name
)

In [3]:
# To be investigated later. Perhaps overkill.

# qpu = QPU(qubits=[qubit], quantum_operations=CustomGeneralOperations)
# qt_platform = QuantumPlatform(setup=device_setup, qpu=qpu)
# device_setup

## Yoko Connection Handling

In [1]:
from yoko_helper import *

[DC1(YokogawaGS200)] Could not connect at TCPIP0:192.168.1.76::inst0
Traceback (most recent call last):
  File "c:\Users\QNL\anaconda3\envs\gridium_ZI\Lib\site-packages\qcodes\instrument\visa.py", line 296, in _connect_and_handle_error
    visa_handle = self._open_resource(address, visalib)
  File "c:\Users\QNL\anaconda3\envs\gridium_ZI\Lib\site-packages\qcodes\instrument\visa.py", line 320, in _open_resource
    resource = resource_manager.open_resource(address)
  File "c:\Users\QNL\anaconda3\envs\gridium_ZI\Lib\site-packages\pyvisa\highlevel.py", line 3265, in open_resource
    info = self.resource_info(resource_name, extended=True)
  File "c:\Users\QNL\anaconda3\envs\gridium_ZI\Lib\site-packages\pyvisa\highlevel.py", line 3190, in resource_info
    ret, _err = self.visalib.parse_resource_extended(
                ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        self.session, resource_name
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\QNL\anaconda3\envs\gridium_ZI\Lib

VI_ERROR_RSRC_NFOUND (-1073807343): Insufficient location information or the requested device or resource is not present in the system.
Connected to: YOKOGAWA GS210 (serial:91P925819, firmware:2.02) in 0.10s


In [ ]:
# --- Changing Currents Manually ---

# - Coil -
# yoko_dict['DC1'].ramp_current(0e-6, 10e-6, 0.02)
# yoko_dict['DC1'].output('on')
# yoko_dict['coil'].source_mode('CURR')
# yoko_dict['DC1'].current_range(0.01)
# print(yoko_dict['coil'].current.get())

# - DC -
yoko_dict['DC2'].ramp_current(0e-6, 10e-6, 0.02)
yoko_dict['DC2'].output('on')
# yoko_dict['dc'].source_mode('CURR')
yoko_dict['DC2'].current_range(0.01)
# print(yoko_dict['dc'].current.get())

KeyError: 'DC1'

# Define Qubits
(May want to move elsewhere after it has been well calibrated)

In [6]:
# TODO: Would be nice to have a variable which keeps track of artificially added attenuation (physically) so that the underlying settings are adjusted appropriately!

# Resonator Frequencies Bookkeeping
# 6.41, 6.575, 6.75, 6.868, 7.198, 7.3825, 7.57


chip_ID = 'GKPv4_1_X2Y7'
# chip_ID = 'BC_Chip_1'

# T_BC = Transmon(
#     uid='T_BC',
#     signals={
#         'acquire': 'T_BC/acquire_line', # These are just dictionary keys for the path to the logical signal group as defined in the YAML
#         'drive': 'T_BC/drive_line', # These could also be key with logical signal rather than key with str
#         'measure': 'T_BC/measure_line',
#     },
#     parameters={
#         'readout_resonator_frequency': 7.6684e9,
#         'readout_lo_frequency': 7.6e9,
#         'resonance_frequency_ge': 4e9, #+ 5.9e6,
#         'drive_lo_frequency': 4e9,
#         'readout_integration_delay': 90e-9, #88e-9,
#         'readout_range_out': 10,
#         'drive_range': -25, 
#         'readout_range_in': 5,
#         'pulse_length': 500e-9,
#         'readout_len': 2e-6,
#         'time_domain_reset_length': 300e-6,
#         'cw_reset_length': 5e-9,
#         'readout_amp': 1,
#         'amplitude_pi': 0.38,
#         'amplitude_pi_div_2': 0.2,
#     }
# )


T0 = Transmon(
    uid='T0',
    signals={
        'acquire': 'T0/acquire_line', # These are just dictionary keys for the path to the logical signal group as defined in the YAML
        'drive': 'T0/drive_line', # These could also be key with logical signal rather than key with str
        'measure': 'T0/measure_line',
    },
    parameters={
        'readout_resonator_frequency': 6.4097e9,
        'readout_lo_frequency': 6.4e9,
        'resonance_frequency_ge': 3.455e9, #+ 5.9e6,
        'drive_lo_frequency': 4e9,
        'readout_integration_delay': 90e-9, #88e-9,
        'readout_range_out': -15,
        'drive_range': 5, 
        'readout_range_in': 5,
        'pulse_length': 500e-9,
        'readout_len': 2e-6,
        'time_domain_reset_length': 300e-6,
        'cw_reset_length': 5e-9,
        'readout_amp': 1,
        'amplitude_pi': 0.38,
        'amplitude_pi_div_2': 0.2,
    }
)

T1 = Transmon(
    uid='T1',
    signals={
        'acquire': 'T1/acquire_line',
        'drive': 'T1/drive_line',
        'measure': 'T1/measure_line',
    },
    parameters={
        'readout_resonator_frequency': 6.5745e9,
        'readout_lo_frequency': 6.6e9,
        'resonance_frequency_ge': 4.069e9,
        'drive_lo_frequency': 4e9,
        'readout_integration_delay': 88e-9,
        'readout_range_out': -15,
        'drive_range': 10, 
        'readout_range_in': 5,
        'pulse_length': 500e-9,
        'readout_len': 2e-6,
        'time_domain_reset_length': 300e-6,
        'cw_reset_length': 5e-9,
        'readout_amp': 1,
        'amplitude_pi': 0.685,
        'amplitude_pi_div_2': 0.39,
    }
)

F0 = Fluxonium(
    uid='F0',
    signals={
        'acquire': 'F0/acquire_line',
        'drive': 'F0/drive_line', 
        'measure': 'F0/measure_line',
        'fast_flux': 'F0/fast_flux',
    },
    parameters={
        'readout_resonator_frequency': 6.7503e9,
        'readout_lo_frequency': 6.8e9,
        'resonance_frequency_ge': 464e6, #454.2e6,
        'drive_lo_frequency': 1e9,
        'readout_integration_delay': 88e-9,
        'readout_range_out': -15,
        'drive_range': -5, #10, 
        'fast_flux_range': None,
        'readout_range_in': -15,
        'pulse_length': 500e-9,
        'readout_len': 2e-6,
        'time_domain_reset_length': 300e-6,
        'cw_reset_length': 5e-9,
        'readout_amp': 0.8,
        'amplitude_pi': 0.9,
        'amplitude_pi_div_2': 0.45,
        'flux_sweetspot': None,#62.4e-6,
        'flux_setpoint': None,
        'res_to_current': None,
    }
)

# try:
#     # fname = f'{data_directory_update()}\\{chip_ID} F0 Flux Sweep Trace'
#     fname = f'data\\2025-10-16\\GKPv4_1_X6Y8 F0 Flux Sweep Trace'
#     # fname = f'data\\2025-06-30\\{chip_ID} F0 Flux Sweep Trace'
#     with open(fname, 'rb') as f:
#         F0.parameters.res_to_current = dill.load(f)
# except Exception as e: print(e)

F1 = Fluxonium(
    uid='F1',
    signals={
        'acquire': 'F1/acquire_line',
        'drive': 'F1/drive_line',
        'measure': 'F1/measure_line',
    },
    parameters={
        'readout_resonator_frequency': 6.8645e9,
        'readout_lo_frequency': 6.8e9,
        'resonance_frequency_ge': 697.5e6,
        'drive_lo_frequency': 0,
        'readout_integration_delay': 88e-9,
        'readout_range_out': -30,
        'drive_range': 10, 
        'fast_flux_range': None,
        'readout_range_in': -15,
        'pulse_length': 500e-9,
        'readout_len': 2e-6,
        'time_domain_reset_length': 300e-6,
        'cw_reset_length': 5e-9,
        'readout_amp': 0.7,
        'amplitude_pi': 0.2,
        'amplitude_pi_div_2': 0.1,
        'flux_sweetspot': None,
        'flux_setpoint': None,
        'res_to_current': None,
    }
)

try:
    # fname = f'{data_directory_update()}\{chip_ID} F1 Flux Sweep Trace'
    # fname = f'data\\2025-08-11\\GKPv4_1_X7Y1 F1 Flux Sweep Trace'
    with open(fname, 'rb') as f:
        F1.parameters.res_to_current = dill.load(f)
except Exception as e: print(e)

GKP1 = Gridium(
    uid='GKP1',
    signals={
        'acquire': 'GKP1/acquire_line',
        'drive': 'GKP1/drive_line',
        'measure': 'GKP1/measure_line',
        'fast_flux': 'GKP1/flux_drive_line',
    },
    parameters={
        'readout_resonator_frequency': 7.197e9,
        'readout_lo_frequency': 7.2e9,
        'resonance_frequency_ge': 2e9,
        'drive_lo_frequency': 2e9,
        'readout_integration_delay': 100e-9,
        'readout_range_out': -15,
        'drive_range': 0, 
        'fast_flux_range': None,
        'readout_range_in': 10,
        'pulse_length': 500e-9,
        'readout_len': 2e-6,
        'time_domain_reset_length': 300e-6,
        'cw_reset_length': 5e-9,
        'readout_amp': 1,
        'amplitude_pi': None,
        'amplitude_pi_div_2': None,
        'flux_sweetspot': None,
        'flux_setpoint': None,
        'res_to_current': None,
    }
)

# try:
#     fname = f'{data_directory_update()}\{chip_ID} GKP1 Flux Sweep Trace'
#     with open(fname, 'rb') as f:
#         GKP1.parameters.res_to_current = dill.load(f)
# except Exception as e: print(e)

GKP2 = Gridium(
    uid='GKP2',
    signals={
        'acquire': 'GKP2/acquire_line',
        'drive': 'GKP2/drive_line', 
        'measure': 'GKP2/measure_line',
    },
    parameters={
        'readout_resonator_frequency': 7.39e9,
        'readout_lo_frequency': 7.4e9,
        'resonance_frequency_ge': 1e9,
        'drive_lo_frequency': 1e9,
        'readout_integration_delay': 88e-9,
        'readout_range_out': 10,
        'drive_range': 5,
        'fast_flux_range': None,
        'readout_range_in': 5,
        'pulse_length': 500e-9,
        'readout_len': 2e-6,
        'time_domain_reset_length': 300e-6,
        'cw_reset_length': 5e-9,
        'readout_amp': 1,
        'amplitude_pi': None,
        'amplitude_pi_div_2': None,
        'flux_sweetspot': None,
        'flux_setpoint': None,
        'res_to_current': None,
    }
)

try:
    fname = f'{data_directory_update()}\{chip_ID} GKP2 Flux Sweep Trace'
    with open(fname, 'rb') as f:
        GKP2.parameters.res_to_current = dill.load(f)
except Exception as e: print(e)

GKP3 = Gridium(
    uid='GKP3',
    signals={
        'acquire': 'GKP3/acquire_line',
        'drive': 'GKP3/drive_line',
        'measure': 'GKP3/measure_line',
    },
    parameters={
        'readout_resonator_frequency': 7.570e9,
        'readout_lo_frequency': 7.6e9,
        'resonance_frequency_ge': 1e9,
        'drive_lo_frequency': 1e9,
        'readout_integration_delay': 88e-9,
        'readout_range_out': -20,
        'drive_range': 10, 
        'fast_flux_range': None,
        'readout_range_in': 5,
        'pulse_length': 500e-9,
        'readout_len': 2e-6,
        'time_domain_reset_length': 300e-6,
        'cw_reset_length': 5e-9,
        'readout_amp': 1,
        'amplitude_pi': None,
        'amplitude_pi_div_2': None,
        'flux_sweetspot': None,
        'flux_setpoint': None,
        'res_to_current': None,
    }
)

try:
    fname = f'{data_directory_update()}\{chip_ID} GKP3 Flux Sweep Trace'
    with open(fname, 'rb') as f:
        GKP3.parameters.res_to_current = dill.load(f)
except Exception as e: print(e)

<string>:233: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<string>:269: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<>:233: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<>:269: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<string>:233: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<string>:269: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<>:233: SyntaxWarning: "\{" is an invalid esca

name 'fname' is not defined
[Errno 2] No such file or directory: 'data\\2026-08-04\\GKPv4_1_X2Y7 GKP2 Flux Sweep Trace'
[Errno 2] No such file or directory: 'data\\2026-08-04\\GKPv4_1_X2Y7 GKP3 Flux Sweep Trace'


C:\Users\QNL\AppData\Local\Temp\ipykernel_31716\517963474.py:233: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
  fname = f'{data_directory_update()}\{chip_ID} GKP2 Flux Sweep Trace'
C:\Users\QNL\AppData\Local\Temp\ipykernel_31716\517963474.py:269: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
  fname = f'{data_directory_update()}\{chip_ID} GKP3 Flux Sweep Trace'


In [7]:
qubit = GKP1

## Experiments

In [8]:
# # exp = global_trace(qubit)

exp = local_trace(
    qubit,
    -5e6,
    5e6,
    # ro_range=-20, #qubit.parameters.readout_range_out,
    averages=2**10,
    trace_pts=101,
    yoko_dict_key='DC1',
    drive_on=False,
)


# exp = punchout(
#     qubit,
#     ro_range_max=10,
#     averages=2**12,
#     lower_power=-1,
#     higher_power=0,
#     rel_ro_left_rf=-20e6, 
#     rel_ro_right_rf=20e6, 
#     ro_pts=101,
#     power_pts=20
# )

# # exp = flux_sweep_trace(
# #     qubit,
# #     'coil',
# #     rel_ro_left_rf=-10e6,
# #     rel_ro_right_rf=10e6,
# #     left_current=-600e-6,
# #     right_current=600e-6,
# #     current_pts=101,
# #     averages=2**12,
# #     trace_pts=51,
# #     silence=False,
# # )

# # exp = full_spectrum(
# #      qubit,
# #     'coil', 
# #     101, 
# #     averages=2**13,
# #     t_delay=10e-8,
# # )

# # exp = sweep_spectrum(
# #     qubit,
# #     'coil', 
# #     -500e6,
# #     500e6, 
# #     101, 
# #     center_drive_freq=4e9,
# #     averages=2**14,
# #     t_delay=1e-6,
# #     drive_length=1100e-9,
# # )

# # exp = flux_sweep_full_spectrum(
# #     qubit,
# #     'coil',
# #     58e-6,
# #     68e-6,
# #     21,
# #     501,
# #     averages=2**11,
# #     t_delay=1e-6
# # )

# # exp = flux_sweep_spectrum(
# #     qubit,
# #     'coil',
# #     60e-6,
# #     64e-6,
# #     8,
# #     -30e6,
# #     30e6,
# #     101,
# #     t_delay=100e-6,
# #     center_drive_freq=0.460e9,
# #     LF_mode=True,
# #     averages=2**12,
# # )

# # exp = dual_flux_sweep(
# #     qubit,
# #     'coil',
# #     'dc',
# #     -100e-6,
# #     100e-6,
# #     201,
# #     -9000e-6,
# #     9000e-6,
# #     201,
# #     averages = 2**15
# # )

# # exp = X90_tuneup(
# #     qubit,
# #     'coil',
# #     lower_amp=0,
# #     upper_amp=1,
# #     amp_count=21,
# #     averages=2**11,
# #     t_drive=200e-9,
# #     t_delay=200e-6,
# # )

# # exp = T1_exp(
# #     qubit,
# #     1e-9,
# #     200e-6,
# #     11,
# #     averages=2**12,
# # )

# # exp = T2_star(
# #     qubit,
# #     1e-9,
# #     10e-6,
# #     31,
# #     averages=2**13,
# #     detuning=-5e6,
# #     reset_delay=200e-6
# # )

# # exp = T2_echo(
# #     qubit,
# #     1e-9,
# #     50e-6,
# #     15,
# #     averages=2**15,
# #     detuning=5.5e6,
# #     reset_delay=100e-6
# # )

Not using current mapping


# Measure

In [9]:
session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,
    experiment=exp)
session.connect();
session.register_neartime_callback(change_current)
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=1000,)

In [11]:
#--- Running Session ---
# yoko_dict['coil'].ramp_current(-200e-6, 10e-6, 0.02)
session.run()
data = session.get_results()
data_results = data.acquired_results['results']

[2026.08.04 14:44:11.419] WARNING SHFQC/QA:dev12247: Channel 0 Output overrange count: 2


ValueError: 3.0 is not a valid DeviceErrorSeverity

In [ ]:
#--- Saving Options ---
hrminsec = datetime.datetime.now().time().strftime(' %H%M%S')
session.save_results(str(data_directory_update()) + '/' + qubit.uid + exp.uid + hrminsec)
# laboneq.simple.save(session, 'my_file_name')
# data.save(str(data_directory_update()) + '/' + qubit.uid + exp.uid)

# Plot

In [ ]:
my_results = session.get_results()
my_acquired_results = my_results.acquired_results['results']
# my_acquired_results.axis_name

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
fig, ax = plot_exp(exp, session, qubit,) #data_type='Phase')
# ax[1].vlines(0.685, -0.6, 0.4)
# fig, ax = plot_exp(exp, session, qubit, data_type='Amplitude') # Punchout Plot
# fig, ax = plot_exp(exp, session, qubit, data_type='Phase') # Punchout Plot

# update_colorbar_limits(fig, new_min=0, new_max=0.1)

exp_analysis(exp, session, qubit, save=True)

# Good debugging statements:
# session.experiment.signals

In [ ]:
from scipy.stats import linregress

# session.experiment.signals
data = my_acquired_results.data
data = data - np.mean(data)
plt.scatter(np.real(data), np.imag(data))
phase = np.unwrap(np.angle(data))
phase = phase - np.mean(phase)
plt.figure()
freqs = my_acquired_results.axis[0]
params = linregress(freqs/1e8, phase)
print(params.slope)
phase_new = phase - freqs/1e8*params.slope - params.intercept
plt.scatter(freqs, phase)
plt.scatter(freqs, freqs/1e8*params.slope + params.intercept)
plt.scatter(freqs, freqs/1e8*(-60) -18)
phase_new = phase_new - np.mean(phase_new)
plt.figure()
plt.scatter(freqs,phase_new)
complex_data = np.abs(data)*np.exp(1j*phase_new)
plt.figure()
plt.scatter(np.real(complex_data), np.imag(complex_data))

In [ ]:
my_results.save('F0_Spectrum_GKP_v4_1_X6Y8')

In [ ]:
freqs = my_acquired_results.axis[0] + exp.signals[f'{qubit.uid}/measure_line'].calibration.local_oscillator.frequency
IQ_data = my_acquired_results.data

In [ ]:

spec_res = my_acquired_results.data

(p_opt, b) = lorentzian.fit(
    freqs,
    abs(IQ_data),
    100e3,
    6.7510e9,
    -1e7,
    1,
    plot=True,
)
opt_freq = p_opt[1]
print(f"Resonant frequency: {opt_freq} Hz")
print(f"Kappa: {p_opt[0]} Hz")
print(f"Amplitude: {p_opt[2]} amplitude (au)")
print(f"Offset: {p_opt[3]} Amplitude (au)")

In [ ]:
def update_colorbar_limits(fig, new_min, new_max):
    """Update colorbar limits for a figure."""
    updated = False
    
    # Method A: Find and update mappable objects
    for ax in fig.get_axes():
        for child in ax.get_children():
            if hasattr(child, 'set_clim'):
                child.set_clim(vmin=new_min, vmax=new_max)
                updated = True
    
    # Method B: If mappables not found, look for colorbar axes
    if not updated:
        for ax in fig.get_axes():
            bbox = ax.get_position()
            if bbox.width < 0.1 or bbox.height < 0.1:  # Likely a colorbar
                if bbox.width < bbox.height:  # Vertical colorbar
                    ax.set_ylim(new_min, new_max)
                else:  # Horizontal colorbar
                    ax.set_xlim(new_min, new_max)
                updated = True
                break
    
    if updated:
        fig.canvas.draw_idle()
    
    return updated

# Usage

## Save/Load current to resonator relationship

In [ ]:
# --- SAVES ---
import dill

fname = f'{data_directory_update()}\{chip_ID} {qubit.uid} {exp.uid}'
with open(fname, 'wb') as f:
    dill.dump(qubit.parameters.res_to_current, f)

In [ ]:
# --- LOADS ---

fname = f'{data_directory_update()}\{chip_ID} {qubit.uid} {exp.uid}'
with open(fname, 'rb') as f:
    loaded_func = dill.load(f)
    print(loaded_func(29e-6))  # Outputs: 8

# Example of Declarative Experiment

In [ ]:
qubit = F1
averages=2**8

readout_pulse = pulse_library.gaussian_square(
    uid=f"readout_pulse_{qubit.uid}",
    length=qubit.parameters.readout_len,
    amplitude=qubit.parameters.readout_amp,
    width=qubit.parameters.readout_len*0.9,
    sigma=0.2,)

readout_lo_freq_sweep = LinearSweepParameter(
    uid='Readout_LO',
    start=6.2e9,
    stop=7.2e9,
    count=2)

readout_freq_sweep = LinearSweepParameter(
    uid='Readout_Frequency',
    start=-500e6,
    stop=499e6,
    count=500)

exp_signals = [
    ExperimentSignal(uid="measure", map_to=lsg[qubit.uid]['measure_line']),
    ExperimentSignal(uid="acquire", map_to=lsg[qubit.uid]['acquire_line']),]

exp = Experiment(
    uid='Global Resonator Trace',
    signals=exp_signals,)
RO_LO_Sweep = Sweep(
    uid='Readout LO Frequency Sweep',
    parameters=readout_lo_freq_sweep)
RT_Loop = AcquireLoopRt(
    uid='Shots',
    count=averages,
    averaging_mode=AveragingMode.CYCLIC,
    acquisition_type=AcquisitionType.SPECTROSCOPY,)
AWG_Freq_Sweep = Sweep(
    uid='Readout Frequency Sweep',
    parameters=readout_freq_sweep,
    reset_oscillator_phase=False)
Meas_Acquire = Section(uid='Pulsed Single Frequency Readout')
Meas_Acquire.play(
    signal='measure',
    pulse=readout_pulse)
Meas_Acquire.acquire(
    signal='acquire', 
    handle='single_freq_data', 
    length=qubit.parameters.readout_len)
Delay_After_Count = Section(uid='Delay Between Readout')

#--- Properly Defining Nesting Order ---
exp.add(RO_LO_Sweep)
RO_LO_Sweep.add(RT_Loop)
RT_Loop.add(AWG_Freq_Sweep)
AWG_Freq_Sweep.add(Meas_Acquire)
AWG_Freq_Sweep.add(Delay_After_Count)
Delay_After_Count.reserve(signal='measure')
Delay_After_Count.reserve(signal='acquire')

#--- Defines Oscillators ---
readout_osc = Oscillator(
    "readout_osc",
    frequency=readout_freq_sweep,
    modulation_type=ModulationType.HARDWARE)
readout_lo = Oscillator(
    "readout_lo",
    frequency=readout_lo_freq_sweep,
    modulation_type=ModulationType.HARDWARE)

#---Calibration object Updates and Application ---
exp_calibration = Calibration()
exp_calibration["measure"] = SignalCalibration(
    oscillator=readout_osc,
    local_oscillator=readout_lo)
exp_calibration["acquire"] = SignalCalibration(
    oscillator=readout_osc,
    local_oscillator=readout_lo)
exp.set_calibration(exp_calibration)

#--- Compiling Session ---
session = Session(
    device_setup=device_setup,
    log_level = logging.WARNING,)
session.connect(use_async_api=True);
compiled_session = session.compile(exp);
# psv.interactive_psv(compiled_session, max_events_to_publish=1000,)

#--- Running Session ---
results = session.run()
my_results = session.get_results() #a deep copy of session.results

# Extracts data from the exp.acquire method with the same key name
my_acquired_results = my_results.acquired_results['single_freq_data']

# HTML tests


<p style='background-color:rgb(255,100,255); color:green;'><em>test </em></p>

<div>
    this is a div
</div>
<h2 style="background-color:DodgerBlue; border:2px solid Tomato">Hello World</h2>
<p style="background-color:Tomato;">Lorem ipsum...</p>

# Notes

There seem to be two ways to go about doing this:

If you are using the (imperative) context based method, you should use the qubit.signals['key'] for the signal

If you are using the declaritive style, you need to manually map the signals

When using AcquisitionType.SPECTROSCOPY, one needs to do two things:
1. Remove one of the local oscillator assignments from the acquire line (probably don't ever need to have assigned it even in integration mode since they share it)
2. MAY need to have at least a 1us delay between rounds (a known issue which generates a QA holdoff error (or in my case a triggering error), though I do not consistently reproduce this error. https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/04_experiment_sequence/tutorials/00_experiment_definition_reference.html

Any time I run a given named experiment, there is basically one way I would
want to plot it, and only a small number of parameters I would like to tweek
(usually none).